# 🦎 Qorvhex Lizard — Documentation Crawler

[![Telegram Channel](https://img.shields.io/badge/Telegram-Channel-blue?style=for-the-badge&logo=telegram)](https://t.me/Qorvhex_Channel) 

Welcome to **Qorvhex Lizard**! A lightning-fast, multithreaded documentation crawler designed to recursively scrape, clean, and format online documentation into a single clean text file for offline reading or feeding into LLMs / RAG pipelines.

---

In [ ]:
# @title ⚙️ تنظیمات استخراج مستندات / Crawler Settings { display-mode: "form" }
START_URL = "https://docs.railway.com" # @param {type:"string"}
OUTPUT_FILE = "documentation_full.txt" # @param {type:"string"}
MAX_WORKERS = 10 # @param {type:"slider", min:1, max:25, step:1}
RESTRICT_TO_PATH = False # @param {type:"boolean"}

import time
import re
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urldefrag
from google.colab import files

# ==========================================
# آماده‌سازی شبکه و پیکربندی خزش
# ==========================================
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

parsed_start = urlparse(START_URL)
base_domain = parsed_start.netloc
base_path = parsed_start.path.rstrip('/')

session = requests.Session()
adapter = requests.adapters.HTTPAdapter(pool_connections=MAX_WORKERS, pool_maxsize=MAX_WORKERS)
session.mount('https://', adapter)
session.mount('http://', adapter)

visited_urls = set()
visited_lock = threading.Lock()
file_lock = threading.Lock()
page_counter = 0
counter_lock = threading.Lock()

def is_valid_url(url):
    parsed = urlparse(url)
    if parsed.netloc != base_domain:
        return False
    if RESTRICT_TO_PATH and not parsed.path.startswith(base_path):
        return False
    ignored_extensions = (
        '.png', '.jpg', '.jpeg', '.gif', '.svg', '.pdf', 
        '.zip', '.tar', '.gz', '.mp4', '.json', '.xml', 
        '.ico', '.webp', '.exe', '.dmg', '.pkg'
    )
    if parsed.path.lower().endswith(ignored_extensions):
        return False
    return True

def extract_clean_content(soup):
    for tag in soup(["script", "style", "nav", "footer", "header", "noscript", 
                     "aside", "svg", "button", "form", "iframe"]):
        tag.extract()

    junk_patterns = [
        "toc", "table-of-contents", "on-this-page", "sidebar", "breadcrumbs",
        "pagination", "feedback", "rating", "menu", "tablist", "tabs",
        "header-nav", "edit-page", "community-links"
    ]
    for element in soup.find_all(attrs={
        "class": lambda c: c and any(p in " ".join(c).lower() for p in junk_patterns)
    }):
        element.extract()
    for element in soup.find_all(attrs={
        "id": lambda i: i and any(p in i.lower() for p in junk_patterns)
    }):
        element.extract()
    for element in soup.find_all(attrs={"role": ["tablist", "tab", "navigation"]}):
        element.extract()

    main_content = (
        soup.find("main") or 
        soup.find("article") or 
        soup.find(class_=lambda c: c and any(w in c.lower() for w in ["content", "markdown", "document", "doc", "body"])) or
        soup.body
    )
    if not main_content:
        return ""

    raw_text = main_content.get_text(separator="\n", strip=True)
    lines = raw_text.splitlines()
    clean_lines = []
    
    junk_line_exact = {
        "on this page", "in this article", "table of contents", 
        "was this page helpful?", "yes", "no", "edit this page",
        "previous", "next", "graphql", "curl", "javascript", "python", "variables"
    }

    for line in lines:
        stripped = line.strip()
        if not stripped or stripped.lower() in junk_line_exact:
            continue
        clean_lines.append(stripped)

    return "\n\n".join(clean_lines)

def process_page(url, file_handle):
    global page_counter
    try:
        response = session.get(url, headers=headers, timeout=10)
        if response.status_code != 200 or 'text/html' not in response.headers.get('Content-Type', ''):
            return []

        soup = BeautifulSoup(response.text, "html.parser")
        raw_title = soup.title.string.strip() if soup.title else "بدون عنوان"
        clean_title = re.sub(r'\s*\|.*$', '', raw_title).strip()
        content = extract_clean_content(soup)

        if content:
            with file_lock:
                file_handle.write("=" * 80 + "\n")
                file_handle.write(f"عنوان: {clean_title}\n")
                file_handle.write(f"آدرس: {url}\n")
                file_handle.write("=" * 80 + "\n\n")
                file_handle.write(content + "\n\n\n")
                file_handle.flush()

        with counter_lock:
            page_counter += 1
            print(f"[{page_counter}] 📄 ذخیره شد: {url}")

        new_links = []
        for link in soup.find_all("a", href=True):
            full_url = urljoin(url, link['href'])
            clean_url, _ = urldefrag(full_url)
            if is_valid_url(clean_url):
                new_links.append(clean_url)

        return new_links
    except Exception as e:
        print(f"⚠️ خطا در بارگذاری {url}: {e}")
        return []

# ==========================================
# شروع اجرای استخراج
# ==========================================
print(f"🦎 شروع استخراج توسط Qorvhex Lizard")
print(f"🚀 آدرس مبدا: {START_URL}")
print(f"⚡ تعداد پردازش همزمان: {MAX_WORKERS}\n")

clean_start_url, _ = urldefrag(START_URL)
to_visit = {clean_start_url}
visited_urls.add(clean_start_url)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        while to_visit:
            futures = {executor.submit(process_page, url, f): url for url in to_visit}
            to_visit = set()
            for future in as_completed(futures):
                discovered_links = future.result()
                with visited_lock:
                    for link in discovered_links:
                        if link not in visited_urls:
                            visited_urls.add(link)
                            to_visit.add(link)

print(f"\n🎉 استخراج کامل شد! مجموع کل صفحات: {page_counter}")
print("📥 در حال دانلود فایل نهایی...")
files.download(OUTPUT_FILE)
